## 1. 환경 설정

- 필요한 모듈
```
pip install google-genai
```
```
pip install chromadb
```

- 파일경로

```
│  chatbot.py # 챗봇 본체
│  STM_to_LTM.py # 데이터 처리 코드
│  
├─demo
│  │  demo_memory_stm.json # 1차 전처리 데이터 파일
│  │  generated_memory_ltm.json # 2차 전처리 데이터 파일
│  │  
│  └─chroma_gemini_handson # chromadb 로컬 저장 파일
└─utils
    │  db_utils.py # 데이터 전처리 사용 모듈
```

- Gemini API 키 발급 필요

- 참고 github : https://github.com/skqorrla/GDG-HandsOn


#### 사용할 입력 파일, 생성 JSON 파일, 전용 Chroma 저장소, Gemini 모델 이름을 한 곳에서 정의합니다. 이후 셀은 이 설정값만 참조하므로 경로와 모델을 바꾸려면 이 셀만 수정하면 됩니다.

In [ ]:
from pathlib import Path

try:
    PROJECT_ROOT = Path(__file__).resolve().parent
except NameError:
    PROJECT_ROOT = Path.cwd()

INPUT_STM_PATH = PROJECT_ROOT / "demo" / "demo_memory_stm.json"
GENERATED_LTM_PATH = PROJECT_ROOT / "demo" / "generated_memory_ltm.json"
CHROMA_STORE_PATH = PROJECT_ROOT / "demo" / "chroma_gemini_handson"
PROMOTION_MODEL = "gemini-3.5-flash"
CHATBOT_MODEL = "gemini-3.5-flash"
EMBEDDING_MODEL = "gemini-embedding-001"

##--------------반드시 입력해주세요!--------------------
GEMINI_API_KEY = "GEMINI_API_KEY"
##------------------------------------------------------

for path in [GENERATED_LTM_PATH]:
    path.parent.mkdir(parents=True, exist_ok=True)
CHROMA_STORE_PATH.mkdir(parents=True, exist_ok=True)
if not INPUT_STM_PATH.exists():
    raise FileNotFoundError(INPUT_STM_PATH)

CONFIG = {
    "input_stm_path": INPUT_STM_PATH,
    "generated_ltm_path": GENERATED_LTM_PATH,
    "chroma_store_path": CHROMA_STORE_PATH,
    "promotion_model": PROMOTION_MODEL,
    "chatbot_model": CHATBOT_MODEL,
    "gemini_api_key_loaded": bool(GEMINI_API_KEY),
}
CONFIG

## 2. STM 입력 확인

`demo/demo_memory_stm.json`을 그대로 읽어 확인합니다. 이후 Gemini가 LTM으로 승격할 입력이 무엇인지 먼저 검증해야, 생성 결과와 저장소 검색 결과가 어떤 기사에서 파생되었는지 추적할 수 있습니다.


In [ ]:
# 입력 STM JSON 구조를 검증하고, LTM으로 승격할 기사 개수를 확인합니다.
import json

try:
    from IPython.display import Markdown, display
except ModuleNotFoundError:
    Markdown = str
    display = print


def validate_stm_payload(payload):
    required_conversation_keys = {"session_id", "articles"}
    required_message_keys = {"id", "memory_type", "session_id", "title", "link", "pubDate", "turn_index", "author", "category"}
    conversations = payload.get("stm_articles")
    if not isinstance(conversations, list) or not conversations:
        raise ValueError("stm_articles는 비어 있지 않은 리스트여야 합니다.")

    message_count = 0
    for conversation_index, conversation in enumerate(conversations):
        missing = required_conversation_keys - conversation.keys()
        if missing:
            raise ValueError(f"conversation[{conversation_index}] 누락 필드: {sorted(missing)}")
        if not isinstance(conversation.get("articles"), list) or not conversation["articles"]:
            raise ValueError(f"conversation[{conversation_index}].articles는 비어 있지 않은 리스트여야 합니다.")
        for message_index, message in enumerate(conversation["articles"]):
            missing = required_message_keys - message.keys()
            if missing:
                raise ValueError(f"message[{conversation_index}:{message_index}] 누락 필드: {sorted(missing)}")
            if message.get("memory_type") != "stm":
                raise ValueError(f"message[{conversation_index}:{message_index}] memory_type은 stm이어야 합니다.")
            if message.get("session_id") != conversation["session_id"]:
                raise ValueError(f"message[{conversation_index}:{message_index}] session_id가 대화와 다릅니다.")
            message_count += 1
    return {"session_count": len(conversations), "message_count": message_count}


stm_data = json.loads(INPUT_STM_PATH.read_text(encoding="utf-8"))
validation_summary = validate_stm_payload(stm_data)
stm_conversations = stm_data["stm_articles"]
message_count = validation_summary["message_count"]

LTM_MEMORY_SCHEMA = {
    "ltm_memory": [
        {
            "id": "string",
            "session_id": "string",
            "summary": "string",
            "topic_tags": ["string"],
            "source_message_ids": ["string"],
            "source_turn_indices": ["integer"],
            "pubDate": "string"
        }
    ]
}

display(Markdown(
    f"**STM 입력 검증 완료**: `{INPUT_STM_PATH.relative_to(PROJECT_ROOT)}`에서 "
    f"세션 {validation_summary['session_count']}개, 메시지 {message_count}개를 읽고 필수 구조를 확인했습니다."
))
display(validation_summary)
display(Markdown("### 목표 LTM JSON 스키마"))
display(LTM_MEMORY_SCHEMA)
display(Markdown("### STM 데이터"))
display(stm_data)

**STM 입력 검증 완료**: `output_news\news_유류세_20260507_1400-1600.json`에서 세션 1개, 메시지 19개를 읽고 필수 구조를 확인했습니다.

{'session_count': 1, 'message_count': 19}

### 목표 LTM JSON 스키마

{'ltm_memory': [{'id': 'string',
   'session_id': 'string',
   'summary': 'string',
   'topic_tags': ['string'],
   'source_message_ids': ['string'],
   'source_turn_indices': ['integer']}]}

### STM 데이터

{'stm_articles': [{'session_id': '3efdddc7',
   'articles': [{'id': '3efdddc7_0',
     'memory_type': 'stm',
     'session_id': '3efdddc7',
     'title': '민주당 &quot;유가보조금 법안 처리에 속도 낼 것&quot;…중동발 물가 대응 총력',
     'link': 'https://www.wolyo.co.kr/news/articleView.html?idxno=311815',
     'description': '이어 &quot;지난 4월 소비자물가가 지난해 같은 달보다 2.6% 상승하며 중동 전쟁 여파가 본격화되고 있지만, 석유 최고가 제도와 <b>유류세</b> 인하 정책이 물가 상승률을 1.2%포인트 낮추는 방파제 역할을 했다&quot;고 평가했다. 이날 한... ',
     'pubDate': '2026-05-07 14:08:00 +0900',
     'turn_index': 0,
     'author': '국민일보',
     'category': '경제'},
    {'id': '3efdddc7_1',
     'memory_type': 'stm',
     'session_id': '3efdddc7',
     'title': '정부, 석유제품 매점매석 금지 2개월 연장…&quot;과징금 신설 추진&quot;',
     'link': 'https://www.asiatoday.co.kr/kn/view.php?key=20260507010001456',
     'description': '정부가 최고가격제와 <b>유류세</b> 인하 효과로 물가가 1.2%포인트(p) 낮아졌다고 추정했지만 3월(2.2%)과 비교하면 0.4%p나 높은 수치다.◇고유가 충격 장기화 대비…민생물가 관리체계 가동이와 관련 정부는 중동전쟁발 고유가... ',
     'pubDate': '2026-05-07 14:12:00 +0900',
     'tu

## 3. Gemini로 LTM 승격

이 단계에서는 `gemini-3.5-flash` 를 호출해 STM 대화에서 장기적으로 보존할 뉴스 기사 요약, 주제 태그를 생성합니다. 승격 결과는 단순 복사가 아니라 다음 검색과 챗봇 응답에 재사용할 수 있는 구조화된 LTM 메모리여야 합니다.

In [ ]:
import json
from google import genai

if not GEMINI_API_KEY:
    raise RuntimeError(".env에 GEMINI_API_KEY 또는 GOOGLE_API_KEY를 설정하세요.")

client = genai.Client(api_key=GEMINI_API_KEY)

stm_data = json.loads(INPUT_STM_PATH.read_text(encoding="utf-8"))

LTM_PROMOTION_INSTRUCTIONS = """
당신은 뉴스기사 STM을 장기 기억 JSON으로 승격하는 메모리 정리자입니다.
반드시 JSON만 반환하고, 각 LTM 항목은 원본 id, source_article_ids, source_turn_indices를 보존하세요.
summary는 장기적으로 재사용할 학습 맥락을 한국어 한두 문장으로 요약하세요.
topic_tags는 검색과 챗봇 응답에 바로 쓸 수 있는 짧은 한국어 배열로 작성하세요.
""".strip()

def build_ltm_promotion_prompt(stm_payload):
    return f"""
{LTM_PROMOTION_INSTRUCTIONS}

[출력 스키마]
{json.dumps(LTM_MEMORY_SCHEMA, ensure_ascii=False, indent=2)}

[입력 STM]
{json.dumps(stm_payload, ensure_ascii=False, indent=2)}
""".strip()

def parse_gemini_json(response):
    text = getattr(response, "text", None) or response.candidates[0].content.parts[0].text
    text = text.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()
    return json.loads(text)


def validate_ltm_payload(payload):
    required_ltm_keys = {"id", "session_id", "summary", "topic_tags", "source_message_ids", "source_turn_indices", "pubDate"}
    memories = payload.get("ltm_memory")
    if not isinstance(memories, list) or not memories:
        raise ValueError("ltm_memory는 비어 있지 않은 리스트여야 합니다.")
    stm_session_ids = {conversation["session_id"] for conversation in stm_conversations}
    stm_message_ids = {message["id"] for conversation in stm_conversations for message in conversation["articles"]}
    for index, item in enumerate(memories):
        missing = required_ltm_keys - item.keys()
        if missing:
            raise ValueError(f"ltm_memory[{index}] 누락 필드: {sorted(missing)}")
        if item["session_id"] not in stm_session_ids:
            raise ValueError(f"ltm_memory[{index}] session_id가 STM 원본에 없습니다.")
        if not str(item["summary"]).strip():
            raise ValueError(f"ltm_memory[{index}] summary는 비어 있을 수 없습니다.")
        for field in [ "topic_tags", "source_message_ids"]:
            if not isinstance(item[field], list) or not all(isinstance(value, str) for value in item[field]):
                raise ValueError(f"ltm_memory[{index}].{field}는 문자열 리스트여야 합니다.")
        if not item["source_message_ids"] or not set(item["source_message_ids"]).issubset(stm_message_ids):
            print(set(item["source_message_ids"]).issubset(stm_message_ids))
            raise ValueError(f"ltm_memory[{index}] source_message_ids가 STM 원본과 맞지 않습니다.")
        if not isinstance(item["source_turn_indices"], list) or not all(isinstance(value, int) for value in item["source_turn_indices"]):
            raise ValueError(f"ltm_memory[{index}].source_turn_indices는 정수 리스트여야 합니다.")
    return payload
    
def validate_generated_ltm_json_structure(payload):
    required_ltm_keys = ["id", "session_id", "summary", "topic_tags", "source_message_ids", "source_turn_indices", "pubDate"]
    memories = payload["ltm_memory"]
    return {
        "root_key_present": "ltm_memory" in payload,
        "ltm_memory_type": type(memories).__name__,
        "ltm_count": len(memories),
        "required_fields": required_ltm_keys,
        "field_presence_by_item": [
            {field: field in item for field in required_ltm_keys}
            for item in memories
        ],
    }

def promote_stm_with_gemini(stm_payload):
    if stm_payload != stm_data:
        raise ValueError("이 핸즈온 셀은 위에서 검증한 STM 입력만 승격합니다.")
    response = client.models.generate_content(model=PROMOTION_MODEL, contents=ltm_promotion_prompt, config={"response_mime_type": "application/json"})
    return response
    # return parse_gemini_json(response)

ltm_promotion_prompt = build_ltm_promotion_prompt(stm_data)
response = promote_stm_with_gemini(stm_data)
payload = parse_gemini_json(response)

generated_ltm_memory = validate_ltm_payload(payload)
ltm_required_field_report = validate_generated_ltm_json_structure(generated_ltm_memory)
ltm_validation_summary = {"ltm_count": len(generated_ltm_memory["ltm_memory"]), "source": INPUT_STM_PATH.relative_to(PROJECT_ROOT).as_posix(), "model": PROMOTION_MODEL}
ltm_validation_summary["required_fields"] = ltm_required_field_report["required_fields"]

# 출력
display(Markdown("### Gemini LTM 승격 프롬프트"))
display(ltm_promotion_prompt[:4000])
display(Markdown("### Gemini 생성 LTM 메모리"))
display(ltm_validation_summary)
display(Markdown("### LTM JSON 구조 검증"))
display(ltm_required_field_report)
display(Markdown("### 변환된 LTM JSON"))
display(generated_ltm_memory)

### Gemini LTM 승격 프롬프트

'당신은 뉴스기사 STM을 장기 기억 JSON으로 승격하는 메모리 정리자입니다.\n반드시 JSON만 반환하고, 각 LTM 항목은 원본 id, source_article_ids, source_turn_indices를 보존하세요.\nsummary는 장기적으로 재사용할 학습 맥락을 한국어 한두 문장으로 요약하세요.\ntopic_tags는 검색과 챗봇 응답에 바로 쓸 수 있는 짧은 한국어 배열로 작성하세요.\n\n[출력 스키마]\n{\n  "ltm_memory": [\n    {\n      "id": "string",\n      "session_id": "string",\n      "summary": "string",\n      "topic_tags": [\n        "string"\n      ],\n      "source_message_ids": [\n        "string"\n      ],\n      "source_turn_indices": [\n        "integer"\n      ]\n    }\n  ]\n}\n\n[입력 STM]\n{\n  "stm_articles": [\n    {\n      "session_id": "3efdddc7",\n      "articles": [\n        {\n          "id": "3efdddc7_0",\n          "memory_type": "stm",\n          "session_id": "3efdddc7",\n          "title": "민주당 &quot;유가보조금 법안 처리에 속도 낼 것&quot;…중동발 물가 대응 총력",\n          "link": "https://www.wolyo.co.kr/news/articleView.html?idxno=311815",\n          "description": "이어 &quot;지난 4월 소비자물가가 지난해 같은 달보다 2.6% 상승하며 중동 전쟁 여파가 본격화되고 있지만, 석유 최고가 제도와 <b

### Gemini 생성 LTM 메모리

{'ltm_count': 1,
 'source': 'output_news/news_유류세_20260507_1400-1600.json',
 'model': 'gemini-2.5-flash',
 'required_fields': ['id',
  'session_id',
  'summary',
  'topic_tags',
  'source_message_ids',
  'source_turn_indices']}

### LTM JSON 구조 검증

{'root_key_present': True,
 'ltm_memory_type': 'list',
 'ltm_count': 1,
 'required_fields': ['id',
  'session_id',
  'summary',
  'topic_tags',
  'source_message_ids',
  'source_turn_indices'],
 'field_presence_by_item': [{'id': True,
   'session_id': True,
   'summary': True,
   'topic_tags': True,
   'source_message_ids': True,
   'source_turn_indices': True}]}

### 변환된 LTM JSON

{'ltm_memory': [{'id': '3efdddc7_LTM_0',
   'session_id': '3efdddc7',
   'summary': '중동 전쟁으로 인한 고유가 상황에 대응하여 정부는 유류세 인하, 유가보조금 지급, 석유 최고가격제 시행 등 다양한 정책으로 물가 안정화를 꾀하고 있습니다. 이러한 노력으로 4월 소비자물가 상승 폭이 약 1.2%p 완화되었으며, 연안 해운업계 등 피해 산업에 대한 긴급 재정 지원도 확대되고 있습니다.',
   'topic_tags': ['고유가',
    '유류세 인하',
    '유가 보조금',
    '물가 안정',
    '정부 정책',
    '경제 대응',
    '중동 전쟁',
    '연안 해운 지원'],
   'source_message_ids': ['3efdddc7_0',
    '3efdddc7_1',
    '3efdddc7_2',
    '3efdddc7_3',
    '3efdddc7_4',
    '3efdddc7_5',
    '3efdddc7_6',
    '3efdddc7_7',
    '3efdddc7_8',
    '3efdddc7_9',
    '3efdddc7_10',
    '3efdddc7_11',
    '3efdddc7_12',
    '3efdddc7_13',
    '3efdddc7_14',
    '3efdddc7_15',
    '3efdddc7_16',
    '3efdddc7_17',
    '3efdddc7_18'],
   'source_turn_indices': [0,
    1,
    2,
    3,
    4,
    5,
    6,
    7,
    8,
    9,
    10,
    11,
    12,
    13,
    14,
    15,
    16,
    17,
    18]}]}

## 4. 생성 JSON 저장 및 확인

이 단계에서는 Gemini가 만든 LTM과 에피소드 메모리를 `generated_memory_ltm.json` 에 저장하고 내용을 표시합니다. 파일로 남겨 두면 API 결과를 재검토하거나 저장소 적재 전후의 데이터 차이를 비교할 수 있습니다.

In [ ]:
GENERATED_LTM_PATH.write_text(json.dumps(generated_ltm_memory, ensure_ascii=False, indent=2), encoding="utf-8")
saved_ltm_memory = json.loads(GENERATED_LTM_PATH.read_text(encoding="utf-8"))
validate_ltm_payload(saved_ltm_memory)
assert GENERATED_LTM_PATH.name == "generated_memory_ltm.json"

ltm_items = saved_ltm_memory["ltm_memory"]


## 5. Chroma 전용 저장소에 영속화

이 단계에서는 생성된 메모리를 Chroma 디렉터리에 저장합니다. 전용 저장소를 사용하면 실습 실행 결과가 기존 데모 데이터와 섞이지 않고, 반복 실행 시 어떤 파일이 산출물인지 명확하게 구분됩니다.


In [ ]:
import chromadb
from chromadb.config import Settings

from utils.db_utils import ensure_ltm_vector_collection

def compact_embedding(text: str) -> list[float]:
    response = client.models.embed_content(model=EMBEDDING_MODEL, contents=text)
    embedding = response.embeddings[0].values
    return [float(value) for value in embedding]

ltm_collection = ensure_ltm_vector_collection(chroma_path=CHROMA_STORE_PATH)
ltm_chroma_ids = [item.get("id") or f"gdg-ltm-{index}" for index, item in enumerate(ltm_items)]

expected_ltm_metadata_by_id = {
    chroma_id: {
        "pubDate": item["pubDate"],
    }
    for chroma_id, item in zip(ltm_chroma_ids, ltm_items)
}

if ltm_items:
    ltm_collection.upsert(
        ids=ltm_chroma_ids,
        documents=[item["summary"] for item in ltm_items],
        embeddings=[compact_embedding(item["summary"]) for item in ltm_items],
        metadatas=[expected_ltm_metadata_by_id[chroma_id] for chroma_id in ltm_chroma_ids],
    )

ltm_chroma_readback = ltm_collection.get(
    ids=ltm_chroma_ids,
    include=["documents", "embeddings", "metadatas"],
) if ltm_items else {"ids": [], "documents": [], "embeddings": [], "metadatas": []}
assert len(ltm_chroma_readback["ids"]) == len(ltm_items)
assert ltm_chroma_readback["documents"] == [item["summary"] for item in ltm_items]
assert len(ltm_chroma_readback["embeddings"]) == len(ltm_items)
ltm_embedding_dimensions = [len(embedding) for embedding in ltm_chroma_readback["embeddings"]]
if ltm_embedding_dimensions:
    assert all(dimension == ltm_embedding_dimensions[0] for dimension in ltm_embedding_dimensions)
ltm_chroma_metadata_by_id = dict(zip(ltm_chroma_readback["ids"], ltm_chroma_readback["metadatas"]))
assert ltm_chroma_metadata_by_id == expected_ltm_metadata_by_id

ltm_chroma_count = ltm_collection.count()
assert ltm_chroma_count >= len(ltm_items)

chroma_validation_summary = {
    "ltm_expected_count": len(ltm_items),
    "ltm_actual_count": ltm_chroma_count,
    "ltm_metadata": ltm_chroma_metadata_by_id,
}


print(chroma_validation_summary)
print({
    "chroma_store_path": CHROMA_STORE_PATH,
    "chroma_sqlite_exists": (CHROMA_STORE_PATH / "chroma.sqlite3").exists(),
    "embedding_model": EMBEDDING_MODEL,
    "ltm_count": ltm_chroma_count,
    "ltm_documents": ltm_chroma_readback["documents"],
    "ltm_embedding_dimensions": ltm_embedding_dimensions,
})